In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

df = pd.read_csv("terremotos.csv")

df.head()
df.info()
df.columns


<class 'pandas.DataFrame'>
RangeIndex: 23412 entries, 0 to 23411
Data columns (total 21 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Date                        23412 non-null  str    
 1   Time                        23412 non-null  str    
 2   Latitude                    23412 non-null  float64
 3   Longitude                   23412 non-null  float64
 4   Type                        23412 non-null  str    
 5   Depth                       23412 non-null  float64
 6   Depth Error                 4461 non-null   float64
 7   Depth Seismic Stations      7097 non-null   float64
 8   Magnitude                   23412 non-null  float64
 9   Magnitude Type              23409 non-null  str    
 10  Magnitude Error             327 non-null    float64
 11  Magnitude Seismic Stations  2564 non-null   float64
 12  Azimuthal Gap               7299 non-null   float64
 13  Horizontal Distance         1604 non-null 

Index(['Date', 'Time', 'Latitude', 'Longitude', 'Type', 'Depth', 'Depth Error',
       'Depth Seismic Stations', 'Magnitude', 'Magnitude Type',
       'Magnitude Error', 'Magnitude Seismic Stations', 'Azimuthal Gap',
       'Horizontal Distance', 'Horizontal Error', 'Root Mean Square', 'ID',
       'Source', 'Location Source', 'Magnitude Source', 'Status'],
      dtype='str')

## Vamos fazer o tratamento de dados

In [ ]:
mediana_rms = df['Root Mean Square'].median()
df['Root Mean Square'] = df['Root Mean Square'].fillna(mediana_rms)

colunas_remover = [
    'ID', 'Source', 'Location Source', 'Magnitude Source', 'Status', 
    'Magnitude Type', 'Depth Error', 'Depth Seismic Stations',
    'Magnitude Error', 'Magnitude Seismic Stations', 
    'Azimuthal Gap', 'Horizontal Distance', 'Horizontal Error'
]

df = df.drop(columns=colunas_remover)

df['Date'] = pd.to_datetime(df['Date'], utc=True, errors='coerce')

df['Year'] = df['Date'].dt.year

df = df.drop(columns=['Date', 'Time'])

print(df['Type'].value_counts())

df = pd.get_dummies(df, columns=['Type'], drop_first=True)


df = df.dropna()

df.head()
df.info()
df.columns

Type
Earthquake           23232
Nuclear Explosion      175
Explosion                4
Rock Burst               1
Name: count, dtype: int64
<class 'pandas.DataFrame'>
Index: 23409 entries, 0 to 23411
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Latitude                23409 non-null  float64
 1   Longitude               23409 non-null  float64
 2   Depth                   23409 non-null  float64
 3   Magnitude               23409 non-null  float64
 4   Root Mean Square        23409 non-null  float64
 5   Year                    23409 non-null  float64
 6   Type_Explosion          23409 non-null  bool   
 7   Type_Nuclear Explosion  23409 non-null  bool   
 8   Type_Rock Burst         23409 non-null  bool   
dtypes: bool(3), float64(6)
memory usage: 1.3 MB


Index(['Latitude', 'Longitude', 'Depth', 'Magnitude', 'Root Mean Square',
       'Year', 'Type_Explosion', 'Type_Nuclear Explosion', 'Type_Rock Burst'],
      dtype='str')

Agora que tratamos os dados, vamos fazer testes de ML. O bloco abaixo divide os dados, treina a Regressão Linear, extrai o RMSE (métrica perrtinente para regressão pedida no projeto) e lista o peso de cada variável

In [ ]:
X = df.drop(columns=['Magnitude'])
y = df['Magnitude']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


modelo = LinearRegression()
modelo.fit(X_train, y_train)


y_pred = modelo.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = modelo.score(X_test, y_test)

print(f"RMSE (Raiz do Erro Quadrático Médio): {rmse:.4f}")
print(f"R² (Score de Explicação): {r2:.4f}\n")


importancia = pd.DataFrame({
    'Variável': X.columns, 
    'Impacto no Modelo': modelo.coef_
}).sort_values(by='Impacto no Modelo', ascending=False)

print(importancia)

RMSE (Raiz do Erro Quadrático Médio): 0.4287
R² (Score de Explicação): 0.0115

                 Variável  Impacto no Modelo
7         Type_Rock Burst           0.337617
3        Root Mean Square           0.177875
0                Latitude           0.000621
2                   Depth           0.000120
1               Longitude           0.000113
4                    Year          -0.000573
6  Type_Nuclear Explosion          -0.049153
5          Type_Explosion          -0.170080
